# Phishing URL Detection Using Machine Learning
**Student:** Hemlata  
**Programme:** IBM SkillsBuild Cyber Security BootCamp 2026 | BharatCares

## Project overview
This educational prototype extracts simple lexical features from URLs and trains a Random Forest classifier to label a URL as *phishing-like* or *legitimate-like*. The included examples are illustrative and small; they are not a validated real-world dataset. Do not use this prototype as the sole basis for security decisions.


## 1. Import libraries
The project uses Python, pandas, and scikit-learn. URL features are extracted with Python's standard library.

In [ ]:
import re
import ipaddress
from urllib.parse import urlparse
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


## 2. Define URL feature extraction

In [ ]:
SUSPICIOUS_WORDS = ('login', 'verify', 'update', 'secure', 'account', 'password', 'banking', 'confirm', 'wallet', 'support')

def extract_features(url):
    """Return basic lexical features for a URL string."""
    raw = str(url).strip()
    candidate = raw if re.match(r'^[a-zA-Z][a-zA-Z0-9+.-]*://', raw) else 'http://' + raw
    parsed = urlparse(candidate)
    hostname = parsed.hostname or ''
    path = parsed.path or ''
    try:
        ipaddress.ip_address(hostname)
        has_ip = 1
    except ValueError:
        has_ip = 0
    lower = raw.lower()
    return {
        'url_length': len(raw),
        'hostname_length': len(hostname),
        'path_length': len(path),
        'dot_count': raw.count('.'),
        'hyphen_count': raw.count('-'),
        'at_count': raw.count('@'),
        'question_count': raw.count('?'),
        'equal_count': raw.count('='),
        'https_scheme': int(parsed.scheme.lower() == 'https'),
        'has_ip_address': has_ip,
        'suspicious_word_count': sum(word in lower for word in SUSPICIOUS_WORDS),
        'has_punycode': int('xn--' in hostname.lower()),
    }


## 3. Prepare an illustrative training dataset
These examples are handcrafted for demonstrating the workflow only. For meaningful evaluation, replace them with a large, labelled, trusted dataset and use a separate test set.

In [ ]:
examples = [
    ('https://www.wikipedia.org', 0), ('https://www.python.org', 0),
    ('https://www.nationalgeographic.com', 0), ('https://www.nhs.uk', 0),
    ('https://www.apple.com', 0), ('https://www.microsoft.com', 0),
    ('https://www.gov.uk', 0), ('https://www.un.org', 0),
    ('https://www.openstreetmap.org', 0), ('https://www.mozilla.org', 0),
    ('https://www.britannica.com', 0), ('https://www.khanacademy.org', 0),
    ('https://www.weather.gov', 0), ('https://www.loc.gov', 0),
    ('https://www.si.edu', 0), ('https://www.noaa.gov', 0),
    ('https://www.who.int', 0), ('https://www.unesco.org', 0),
    ('https://www.djangoproject.com', 0), ('https://pandas.pydata.org', 0),
    ('http://account-verify-login.example.com/update', 1),
    ('http://secure-bank-login.example.net/confirm?account=1', 1),
    ('http://192.0.2.10/verify/password', 1),
    ('http://wallet-support.example.org/secure-login', 1),
    ('http://example.com@malicious.example.net/login', 1),
    ('http://xn--secure-example-9k2.example/verify', 1),
    ('http://account-update.example.com/password?confirm=yes', 1),
    ('http://banking-security.example.net/login/account', 1),
    ('http://verify-user.example.org/secure/update', 1),
    ('http://support-wallet.example.com/confirm-login', 1),
    ('http://203.0.113.8/secure/account', 1),
    ('http://login.example.net/verify?password=reset', 1),
    ('http://account-security.example.org/update-confirm', 1),
    ('http://secure-check.example.com/banking/login', 1),
    ('http://user-confirm.example.net/wallet/support', 1),
    ('http://verify-account.example.org/login?update=1', 1),
    ('http://helpdesk-secure.example.com/password/confirm', 1),
    ('http://bank-login.example.net/account/verify', 1),
    ('http://update-wallet.example.org/secure-login', 1),
    ('http://account.example.com/confirm?verify=1', 1),
    ('http://security-check.example.net/login/update', 1),
]
df = pd.DataFrame(examples, columns=['url', 'label'])
df['label_name'] = df['label'].map({0: 'Legitimate-like', 1: 'Phishing-like'})
display(df.head())
print('Illustrative records:', len(df))
print(df['label_name'].value_counts())


## 4. Train and evaluate the model

In [ ]:
feature_df = pd.DataFrame([extract_features(url) for url in df['url']])
X_train, X_test, y_train, y_test = train_test_split(
    feature_df, df['label'], test_size=0.25, random_state=42, stratify=df['label']
)
model = RandomForestClassifier(n_estimators=150, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print('Accuracy on this small illustrative split:', round(accuracy_score(y_test, predictions), 3))
print('\nClassification report:\n', classification_report(y_test, predictions, target_names=['Legitimate-like', 'Phishing-like'], zero_division=0))
print('Confusion matrix (rows=true, columns=predicted):\n', confusion_matrix(y_test, predictions))
print('\nImportant limitation: this score is not a real-world performance estimate because the sample is tiny and handcrafted.')


## 5. Test a URL

In [ ]:
def predict_url(url):
    """Predict a URL's class and return a cautious educational warning."""
    row = pd.DataFrame([extract_features(url)])
    prediction = int(model.predict(row)[0])
    label = 'Phishing-like / suspicious' if prediction == 1 else 'Legitimate-like'
    probabilities = model.predict_proba(row)[0]
    confidence = float(probabilities[list(model.classes_).index(prediction)])
    return {'url': url, 'prediction': label, 'model_score': round(confidence, 3),
            'notice': 'This is a demonstration model, not a guarantee of safety.'}

test_urls = [
    'https://www.wikipedia.org',
    'http://account-verify-login.example.com/update?password=1'
]
for item in test_urls:
    print(predict_url(item))


## 6. Conclusions and future improvements
- URL lexical features can help demonstrate how machine learning may assist phishing awareness.
- The current training examples are synthetic and too small for operational security use.
- A stronger version should use a large, current, labelled dataset; deduplicate domains; split data by domain/time to reduce leakage; evaluate precision, recall, F1 and false-positive rate; and document data provenance.
- Do not open suspicious links to test them. Use approved security tools and follow institutional guidance.
